# TGB Modelling

## Load the Full Dataset
The full dataset is 1.2Gb as a parquet file, with 700+ columns. Care is needed to ensure memory constraints are respected. 

In [1]:
import pandas as pd
import gc

df_full = pd.read_parquet("../Data/EastAfrica Preprocessed/eastafrica-ready-for-tgb.parquet")

In [2]:
#convert float64s to float32 to save memory:
float_cols = df_full.select_dtypes(include=["float64"]).columns

df_full[float_cols] = df_full[float_cols].astype("float32")


In [3]:
print("Shape: ", df_full.shape)

for col in df_full.columns:
    print (f"{col}: ", df_full[col].dtype)


Shape:  (2314906, 483)
id:  category
name:  category
country:  category
latitude:  float32
longitude:  float32
minBorderDistanceKm:  float32
minCapitalDistanceKm:  float32
periodStart:  datetime64[ns]
month_sin:  float32
month_cos:  float32
is_voting:  bool
TargetConflictLocal_t+1:  boolean
TargetConflictRegional_t+1:  boolean
TargetConflictLocal_t+3:  boolean
TargetConflictRegional_t+3:  boolean
TargetConflictLocal_t+6:  boolean
TargetConflictRegional_t+6:  boolean
TargetConflictLocal_t+12:  boolean
TargetConflictRegional_t+12:  boolean
LocalEventCount_std2:  float32
LocalEventCount_std3:  float32
LocalEventCount_std6:  float32
LocalEventCount_std12:  float32
LocalEventCount_std18:  float32
LocalDistinctActorCount_std2:  float32
LocalDistinctActorCount_std3:  float32
LocalDistinctActorCount_std6:  float32
LocalDistinctActorCount_std12:  float32
LocalDistinctActorCount_std18:  float32
LocalDistinctEventTypes_std2:  float32
LocalDistinctEventTypes_std3:  float32
LocalDistinctEventTypes_

In [4]:
#define target columns, we'll iterate over this list later. These need to be exact matches to stop target information leaking into the training (looking at you, AUC = 1) 
target_columns = [
"TargetConflictLocal_t+1",
"TargetConflictRegional_t+1",
"TargetConflictLocal_t+3",
"TargetConflictRegional_t+3",
"TargetConflictLocal_t+6",
"TargetConflictRegional_t+6",
"TargetConflictLocal_t+12",
"TargetConflictRegional_t+12"
]

The below builds a list of columns we will train with, we can know these in advance: most of the columns are lagged, a small number (country, latitude, longitude, country, minBorderDistanceKm, minCapitalDistanceKm, is_voting) are either static for the place, or can be known in advance)

In [5]:
#Drop columns named median: suspect leakage.
df_full = df_full.drop(columns=df_full.filter(regex="Median").columns)
df_full = df_full.drop(columns=df_full.filter(regex="median").columns)

In [6]:

feature_cols = [
    c for c in df_full.columns
    if c not in target_columns
    and c not in ["id", 'name',"periodStart", "periodEnd", "period_midpoint"]
]
print(feature_cols)

['country', 'latitude', 'longitude', 'minBorderDistanceKm', 'minCapitalDistanceKm', 'month_sin', 'month_cos', 'is_voting', 'LocalEventCount_std2', 'LocalEventCount_std3', 'LocalEventCount_std6', 'LocalEventCount_std12', 'LocalEventCount_std18', 'LocalDistinctActorCount_std2', 'LocalDistinctActorCount_std3', 'LocalDistinctActorCount_std6', 'LocalDistinctActorCount_std12', 'LocalDistinctActorCount_std18', 'LocalDistinctEventTypes_std2', 'LocalDistinctEventTypes_std3', 'LocalDistinctEventTypes_std6', 'LocalDistinctEventTypes_std12', 'LocalDistinctEventTypes_std18', 'LocalDistinctEventSubtypes_std2', 'LocalDistinctEventSubtypes_std3', 'LocalDistinctEventSubtypes_std6', 'LocalDistinctEventSubtypes_std12', 'LocalDistinctEventSubtypes_std18', 'LocalTotalFatalities_std2', 'LocalTotalFatalities_std3', 'LocalTotalFatalities_std6', 'LocalTotalFatalities_std12', 'LocalTotalFatalities_std18', 'LocalMeanSeverity_std2', 'LocalMeanSeverity_std3', 'LocalMeanSeverity_std6', 'LocalMeanSeverity_std12', 'L

## Define Training and testing datasets
Tree based models do not require a full series (of Ids) to be provided, or for full series to be present in the training data: we can split as required. 

### Removing records with NaN
We can drop the first 18 periods, as they will have NaN for the longest window. 

In [7]:
print("Shape before dropping: ", df_full.shape)
df_full = df_full[df_full['LocalEventCount_std18'].notna()].copy()
print("Shape after dropping: ", df_full.shape)

Shape before dropping:  (2314906, 483)
Shape after dropping:  (2298264, 483)


We can also drop the last 12 periods. These will have False or NaN as their value in the t+12 target columns. That's not useful for actually comparing the performance of the models. 

In [8]:
print("Shape before dropping: ", df_full.shape)
all_period_starts = sorted(df_full['periodStart'].unique(), reverse = True)
invalid_target_periods = all_period_starts[:12]
print("Invalid periods: ",invalid_target_periods )
df_full = df_full[~df_full['periodStart'].isin(invalid_target_periods)].copy()
print("Shape after dropping: ", df_full.shape)


Shape before dropping:  (2298264, 483)
Invalid periods:  [Timestamp('2025-07-31 00:00:00'), Timestamp('2025-07-03 00:00:00'), Timestamp('2025-06-05 00:00:00'), Timestamp('2025-05-08 00:00:00'), Timestamp('2025-04-10 00:00:00'), Timestamp('2025-03-13 00:00:00'), Timestamp('2025-02-13 00:00:00'), Timestamp('2025-01-16 00:00:00'), Timestamp('2024-12-19 00:00:00'), Timestamp('2024-11-21 00:00:00'), Timestamp('2024-10-24 00:00:00'), Timestamp('2024-09-26 00:00:00')]
Shape after dropping:  (2098416, 483)


### Train/Test Split


#### Build the Testing Set

In [9]:
all_period_starts = sorted(df_full['periodStart'].unique(), reverse = True)
testing_periods = all_period_starts[:12]
print("Testing periods: ", min(testing_periods) , " - ", max(testing_periods))
df_test = df_full[df_full['periodStart'].isin(testing_periods)].copy()
print("Testing shape:", df_test.shape)

Testing periods:  2023-10-26 00:00:00  -  2024-08-29 00:00:00
Testing shape: (199848, 483)


#### Buid the Training set

In [10]:
df_train = df_full[~df_full["periodStart"].isin(testing_periods)].copy()
print("Training shape: ", df_train.shape)
unique_training_periods = df_train['periodStart'].unique()
print("Training periods: ", min(unique_training_periods), ' - ',  max(unique_training_periods))

Training shape:  (1898568, 483)
Training periods:  2015-01-01 00:00:00  -  2023-09-28 00:00:00


In [11]:
del df_full
gc.collect()

29

In [30]:
df_test.to_parquet('../Data/TGB Preprocessed/df_test.parquet')

In [31]:
df_train.to_parquet('../Data/TGB Preprocessed/df_train.parquet')

## LightGBM Training

### Subsampling strategy for 5-fold CV
Every period contributes to the sample. temporal structure is preserved, fold size is controlled at 200K, positives are kept, neagtives are balances across time. All positives are selected, which might not be a good idea since they are much more common than when doing point predictions. 

In [12]:
for col in feature_cols:
    print (col)

country
latitude
longitude
minBorderDistanceKm
minCapitalDistanceKm
month_sin
month_cos
is_voting
LocalEventCount_std2
LocalEventCount_std3
LocalEventCount_std6
LocalEventCount_std12
LocalEventCount_std18
LocalDistinctActorCount_std2
LocalDistinctActorCount_std3
LocalDistinctActorCount_std6
LocalDistinctActorCount_std12
LocalDistinctActorCount_std18
LocalDistinctEventTypes_std2
LocalDistinctEventTypes_std3
LocalDistinctEventTypes_std6
LocalDistinctEventTypes_std12
LocalDistinctEventTypes_std18
LocalDistinctEventSubtypes_std2
LocalDistinctEventSubtypes_std3
LocalDistinctEventSubtypes_std6
LocalDistinctEventSubtypes_std12
LocalDistinctEventSubtypes_std18
LocalTotalFatalities_std2
LocalTotalFatalities_std3
LocalTotalFatalities_std6
LocalTotalFatalities_std12
LocalTotalFatalities_std18
LocalMeanSeverity_std2
LocalMeanSeverity_std3
LocalMeanSeverity_std6
LocalMeanSeverity_std12
LocalMeanSeverity_std18
LocalProtestersEventCount_std2
LocalProtestersEventCount_std3
LocalProtestersEventCount_st

In [13]:
def make_year_blocked_folds(df, year_col="periodStart", n_folds=None):
    """
    Forward-chaining CV where each fold trains on one year and validates on the next.
    Assumes periodStart encodes 28-day periods that map to calendar years.
    """
    # Extract year from periodStart
    df["year"] = pd.to_datetime(df[year_col]).dt.year

    years = sorted(df["year"].unique())

    folds = []

    for i in range(len(years) - 1):
        train_year = years[i]
        val_year   = years[i + 1]

        train_idx = df[df["year"] == train_year].index
        val_idx   = df[df["year"] == val_year].index

        folds.append((train_idx, val_idx))

    return folds


In [14]:
folds = make_year_blocked_folds(df_train, year_col="periodStart")
#folds

In [15]:
gc.collect()

24

## Known good code for training LightGBM models. 

In [16]:
import lightgbm as lgb
import joblib
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import os

#folds = make_time_series_folds(df_train, "periodStart", n_folds=2)

models = {}
cv_results = {}

feature_cols = [
    c for c in df_train.columns
    if c not in target_columns
    and c not in ["id", "name", 'year', "periodStart", 'latitude', 'longitude', "periodEnd", "period_midpoint"]
]

for target_column in target_columns:

    print(f"\n=== Training {target_column} with Time-Series CV ===")

        #Skip iteration if the model already exists
    if os.path.isfile(f"../Models/CatBoost/catboost_model_{target_column}.pkl"):
        print("Model already exists, skipping...")
        break
    
    fold_metrics = []

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

        gc.collect() #garbage collection, no silver bullet but might help RAM issues.
        
        print(f"\nFold {fold_num}")

        # Full fold (past only)
        #df_train_fold = df_train.loc[train_idx]

        # Time-aware subsample to ~200k rows
        #df_train_fold_small = subsample_time_period_balanced(
        #    df_train_fold,
        #    target_column,
        #    max_rows=100_000
        #)

        #y_train = df_train_fold_small[target_column]
        #X_train = df_train_fold_small[feature_cols]

        # Validation stays full
        #df_val_fold = df_train.loc[val_idx]
        #y_val = df_val_fold[target_column]
        #X_val = df_val_fold[feature_cols]

        X_train = df_train.loc[train_idx, feature_cols]
        y_train = df_train.loc[train_idx, target_column]

        X_val = df_train.loc[val_idx, feature_cols]
        y_val = df_train.loc[val_idx, target_column]

        
        # Compute class imbalance weight
        #positives = int(y_train.sum())
        #negatives = len(y_train) - positives
        #scale_pos_weight = max(negatives / positives if positives > 0 else 1.0, 1.0)

        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
        print("Class weights:", class_weight_dict)
        
        #print(f"scale_pos_weight={scale_pos_weight}, "f"train_rows={len(y_train)}, val_rows={len(y_val)}")

        # Build model
        model = lgb.LGBMClassifier(
            boosting_type='gbdt',
            objective="binary",
            learning_rate=0.01,
            n_estimators=100,   # smaller for CV
            #scale_pos_weight=scale_pos_weight,
            class_weight=class_weight_dict,
            random_state=1,
            n_jobs=5,
        )

        # Train on this fold
        model.fit(X_train, y_train)

        # Predict on validation fold
        y_pred = model.predict_proba(X_val)[:, 1]

        # Compute metric
        auc = roc_auc_score(y_val, y_pred)
        fold_metrics.append(auc)

        print(f"Fold {fold_num} AUC: {auc:.4f}")

        #feature importances: useful for identifying leakage.
        importances = model.feature_importances_
        feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
        })
        
        feat_imp = feat_imp.sort_values("importance", ascending=False)
        top20 = feat_imp.head(20)
        print(top20)

        
        # Free memory
        #del df_train_fold, df_train_fold_small, df_val_fold
        del X_train, X_val, y_train, y_val, y_pred
        gc.collect()

    #end of fold loop
    
    # Store CV results
    cv_results[target_column] = {
        "mean_auc": np.mean(fold_metrics),
        "folds": fold_metrics
    }

    print(f"\n{target_column} CV mean AUC: {np.mean(fold_metrics):.4f}")

    # === Train final model on ALL df_train ===

    # Build final training set: all periods, subsampled by per-period quota
    #df_train_final = subsample_time_period_balanced(
    #    df_train,
    #    target_column,
    #    max_rows=200_000
    #)

    y_train_final = df_train[target_column]
    X_train_final = df_train[feature_cols]
    
    y_train_full = df_train[target_column]
   # positives = int(y_train_full.sum())
   # negatives = len(y_train_full) - positives
    # = max(negatives / positives if positives > 0 else 1.0, 1.0)

    X_train_full = df_train[feature_cols]

    classes = np.unique(y_train_full)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
    class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
    print("Class weights:", class_weight_dict)
    
    final_model = lgb.LGBMClassifier(
        boosting_type='gbdt',
        objective="binary",
        learning_rate=0.01,
        n_estimators=500,
        #scale_pos_weight=scale_pos_weight,
        class_weight=class_weight_dict,
        random_state=1,
        n_jobs=5,
    )

    final_model.fit(X_train_full, y_train_full)

    joblib.dump(final_model, f"../Models/LightGBM/lightgbm_model_{target_column}.pkl")
    models[target_column] = final_model

    del X_train_full, y_train_full#, df_train_final


=== Training TargetConflictLocal_t+1 with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(0.5297574772940808), np.True_: np.float64(8.901249794441704)}
[LightGBM] [Info] Number of positive: 12162, number of negative: 204352
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.144370 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 58195
[LightGBM] [Info] Number of data points in the train set: 216514, number of used features: 454
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Fold 1 AUC: 0.9985
                                     feature  importance
176                   LocalMeanSeverity_lag1         464
339                  LocalMeanSeverity_mean2         419
334               LocalTotalFatalities_mean2         418
31                   

In [17]:
gc.collect()

53

## Random Forest Models

In [26]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

models_rf = {}
cv_results = {}

feature_cols = [
    c for c in df_train.columns
    if c not in target_columns
    and c not in ["id", "name", 'year', "periodStart", 'latitude', 'longitude', "periodEnd", "period_midpoint"]
]

for target_column in target_columns:

    print(f"\n=== Training {target_column} for Random Forest with Time-Series CV ===")

    #Skip iteration if the model already exists
    if os.path.isfile(f"../Models/RF/rf_model_{target_column}.pkl"):
        print("Model already exists, skipping...")
        break
    
    fold_metrics = []

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

        gc.collect() #garbage collection, no silver bullet but might help RAM issues.
        
        print(f"\nFold {fold_num}")


        X_train = df_train.loc[train_idx, feature_cols]
        X_train['country'] = X_train['country'].astype('category').cat.codes #need to shift categorical col to the categorical codes; these will match the exported ones. 
        y_train = df_train.loc[train_idx, target_column]

        X_val = df_train.loc[val_idx, feature_cols]
        X_val['country'] = X_val['country'].astype('category').cat.codes
        y_val = df_train.loc[val_idx, target_column]
        
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
        print("Class weights:", class_weight_dict)

        #build model
        model_rf = RandomForestClassifier(
            n_estimators=100,      # number of trees
            max_depth=None,       # let trees grow fully
            random_state=42,      # reproducibility
            n_jobs=5,            # use all CPU cores
            class_weight=class_weight_dict, 
        )

        # Train on this fold
        model_rf.fit(X_train, y_train)

        # Predict on validation fold
        y_pred = model_rf.predict_proba(X_val)[:, 1]

        # Compute metric
        auc = roc_auc_score(y_val, y_pred)
        fold_metrics.append(auc)

        print(f"Fold {fold_num} AUC: {auc:.4f}")

        #feature importances: useful for identifying leakage.
        importances = model_rf.feature_importances_
        feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
        })
        
        feat_imp = feat_imp.sort_values("importance", ascending=False)
        top20 = feat_imp.head(20)
        print(top20)

        
        # Free memory
        #del df_train_fold, df_train_fold_small, df_val_fold
        del X_train, X_val, y_train, y_val, y_pred
        gc.collect()

    #end of fold loop
    
    # Store CV results
    cv_results[target_column] = {
        "mean_auc": np.mean(fold_metrics),
        "folds": fold_metrics
    }

    print(f"\n{target_column} CV mean AUC: {np.mean(fold_metrics):.4f}")
    
    y_train_full = df_train[target_column]
    X_train_full = df_train[feature_cols]
    X_train_full['country'] = X_train_full['country'].astype('category').cat.codes

    classes = np.unique(y_train_full)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
    class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
    print("Class weights:", class_weight_dict)
    
    final_model_rf = RandomForestClassifier(
        n_estimators=200,      # number of trees
        max_depth=None,       # let trees grow fully
        random_state=42,      # reproducibility
        n_jobs=5,            # use all CPU cores
        class_weight=class_weight_dict, 
    )

    final_model_rf.fit(X_train_full, y_train_full)

    joblib.dump(final_model, f"../Models/RF/rf_model_{target_column}.pkl")
    models_rf[target_column] = final_model_rf

    importances = final_model_rf.feature_importances_
    feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
    })
    
    del X_train_full, y_train_full


=== Training TargetConflictLocal_t+1 for Random Forest with Time-Series CV ===
Model already exists, skipping...


In [32]:
gc.collect()

8846

## CatBoost Modelling

In [33]:
!pip install catboost
from catboost import CatBoostClassifier
import joblib
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

models_catboost = {}
cv_results = {}

feature_cols = [
    c for c in df_train.columns
    if c not in target_columns
    and c not in ["id", "name", 'year', 'latitude', 'longitude', "periodStart", "periodEnd", "period_midpoint"]
]

for target_column in target_columns:

    print(f"\n=== Training {target_column} for Random Forest with Time-Series CV ===")

    #Skip iteration if the model already exists
    if os.path.isfile(f"../Models/CatBoost/catboost_model_{target_column}.pkl"):
        print("Model already exists, skipping...")
        break
    
    
    fold_metrics = []

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

        gc.collect() #garbage collection, no silver bullet but might help RAM issues.
        
        print(f"\nFold {fold_num}")


        X_train = df_train.loc[train_idx, feature_cols]
       # X_train['country'] = X_train['country'].astype('category').cat.codes #need to shift categorical col to the categorical codes; these will match the exported ones. 
        y_train = df_train.loc[train_idx, target_column]

        X_val = df_train.loc[val_idx, feature_cols]
        #X_val['country'] = X_val['country'].astype('category').cat.codes
        y_val = df_train.loc[val_idx, target_column]

        cat_features = [i for i, col in enumerate(X_train.columns)
                if X_train[col].dtype.name in ["category", "object"]]
        
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
        class_weights = [class_weight_dict[cls] for cls in classes]
        print("Class weights:", class_weight_dict)

        #build model
        model_catboost = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=8,
            loss_function="Logloss",
            eval_metric="AUC",
            cat_features=cat_features,
            class_weights = class_weights,
            verbose=False
        )
        # Train on this fold
        model_catboost.fit(X_train, y_train)

        # Predict on validation fold
        y_pred = model_catboost.predict_proba(X_val)[:, 1]

        # Compute metric
        auc = roc_auc_score(y_val, y_pred)
        fold_metrics.append(auc)

        print(f"Fold {fold_num} AUC: {auc:.4f}")

        #feature importances: useful for identifying leakage.
        importances = model_catboost.feature_importances_
        feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
        })
        
        feat_imp = feat_imp.sort_values("importance", ascending=False)
        top20 = feat_imp.head(20)
        print(top20)

        
        # Free memory
        #del df_train_fold, df_train_fold_small, df_val_fold
        del X_train, X_val, y_train, y_val, y_pred
        gc.collect()

    #end of fold loop
    
    # Store CV results
    cv_results[target_column] = {
        "mean_auc": np.mean(fold_metrics),
        "folds": fold_metrics
    }

    print(f"\n{target_column} CV mean AUC: {np.mean(fold_metrics):.4f}")
    
    y_train_full = df_train[target_column]
    X_train_full = df_train[feature_cols]
   # X_train_full['country'] = X_train_full['country'].astype('category').cat.codes
    cat_features = [i for i, col in enumerate(X_train_full.columns)
                if X_train_full[col].dtype.name in ["category", "object"]]
    
    classes = np.unique(y_train_full)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
    class_weight_dict = {cls: w for cls, w in zip(classes, weights)}
    class_weights = [class_weight_dict[cls] for cls in classes]
    print("Class weights:", class_weight_dict)


    
    final_model_catboost = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=8,
            loss_function="Logloss",
            eval_metric="AUC",
            cat_features=cat_features,
            class_weights = class_weights,
            verbose=False,
        )

    final_model_catboost.fit(X_train_full, y_train_full)

    joblib.dump(final_model_catboost, f"../Models/CatBoost/catboost_model_{target_column}.pkl")
    models_catboost[target_column] = final_model_catboost

    importances = final_model_catboost.feature_importances_
    feat_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances
    })
    feat_imp = feat_imp.sort_values("importance", ascending=False)
    top20 = feat_imp.head(20)
    print(top20)

    del X_train_full, y_train_full


=== Training TargetConflictLocal_t+1 for Random Forest with Time-Series CV ===

Fold 1
Class weights: {np.False_: np.float64(0.5297574772940808), np.True_: np.float64(8.901249794441704)}
Fold 1 AUC: 0.9998
                                  feature  importance
339               LocalMeanSeverity_mean2   22.436378
176                LocalMeanSeverity_lag1   21.595074
31                 LocalMeanSeverity_std2   10.682061
334            LocalTotalFatalities_mean2    9.371958
170             LocalTotalFatalities_lag1    5.469648
26              LocalTotalFatalities_std2    5.389215
324         LocalDistinctEventTypes_mean2    3.643776
340               LocalMeanSeverity_mean3    1.732453
100         RegionalTotalFatalities_std18    1.489556
27              LocalTotalFatalities_std3    1.132089
32                 LocalMeanSeverity_std3    0.763343
335            LocalTotalFatalities_mean3    0.623352
113    RegionalStateForcesEventCount_std6    0.560424
437    RegionalCiviliansEventCount_me

In [ ]:
gc.collect()